In [1]:
import sys
import platform

print("Python:", sys.version)
print("Ejecutable:", sys.executable)
print("Sistema:", platform.platform())

Python: 3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]
Ejecutable: C:\Users\Christian\.conda\envs\tfm-piloto\python.exe
Sistema: Windows-10-10.0.26200-SP0


In [2]:
import fastapi
import gradio
import langgraph
import faiss
import pypdf

print("Dependencias verificadas correctamente")

Dependencias verificadas correctamente


In [3]:
from app.core.config import settings

clave_configurada = bool(
    settings.openai_api_key
    and settings.openai_api_key.get_secret_value()
)

print("Modelo configurado:", settings.openai_model)
print("Clave configurada:", clave_configurada)

Modelo configurado: gpt-4.1-mini
Clave configurada: True


In [4]:
from pathlib import Path
from dotenv import dotenv_values

ruta_env = Path.cwd() / ".env"
variables = dotenv_values(ruta_env)
clave = variables.get("OPENAI_API_KEY", "")

print("Carpeta actual:", Path.cwd())
print("Archivo .env encontrado:", ruta_env.exists())
print("Variable encontrada:", bool(clave))

Carpeta actual: C:\Users\Christian\Documents\tfm-piloto
Archivo .env encontrado: True
Variable encontrada: True


In [5]:
from openai import OpenAI

from app.core.config import settings


client = OpenAI(
    api_key=settings.openai_api_key.get_secret_value()
)

response = client.responses.create(
    model=settings.openai_model,
    input="Responde únicamente con: conexión correcta",
    max_output_tokens=20,
)

print(response.output_text)

conexión correcta


In [6]:
from app.services.llm_service import LLMService

llm = LLMService()
respuesta = llm.generar_respuesta("Explica en una frase qué es un asistente consular inteligente.")
print(respuesta)

Un asistente consular inteligente es una herramienta digital que utiliza inteligencia artificial para brindar información y apoyo automatizado a ciudadanos sobre trámites y servicios consulares.


In [7]:
from app.rag.document_loader import cargar_documentos_txt

documentos = cargar_documentos_txt()

print("Documentos cargados:", len(documentos))
print("Archivo:", documentos[0]["archivo"])
print("Primeros caracteres:")
print(documentos[0]["contenido"][:300])

Documentos cargados: 3
Archivo: dni_consulta_estado.txt
Primeros caracteres:
Fuente: https://www.consulado.pe/es/turin/Paginas/Retiro-de-Documentos.aspx#dni
Fecha de consulta: 30/06/2026
Ámbito: Consulta de estado de trámite de DNI realizado en oficina consular

Trámite: Consulta de estado de trámite de DNI.

Este documento corresponde a consultas de ciudadanos que ya realiz


In [8]:
from app.rag.document_loader import cargar_documentos_txt
from app.rag.text_splitter import dividir_documentos

documentos = cargar_documentos_txt()
fragmentos = dividir_documentos(documentos)

print("Documentos:", len(documentos))
print("Fragmentos:", len(fragmentos))

for f in fragmentos:
    print("----")
    print(f["id"])
    print(f["contenido"][:500])

Documentos: 3
Fragmentos: 14
----
dni_consulta_estado.txt::chunk_0
Fuente: https://www.consulado.pe/es/turin/Paginas/Retiro-de-Documentos.aspx#dni
Fecha de consulta: 30/06/2026
Ámbito: Consulta de estado de trámite de DNI realizado en oficina consular

Trámite: Consulta de estado de trámite de DNI.

Este documento corresponde a consultas de ciudadanos que ya realizaron un trámite de DNI en una oficina consular y desean verificar el avance, disponibilidad o estado de entrega de su documento.

Cuándo usar esta información:
- Cuando el ciudadano indica que ya real
----
dni_consulta_estado.txt::chunk_1
Consulta de estado:
- El ciudadano puede hacer seguimiento al estado de su trámite de DNI en el portal de RENIEC.
- URL oficial de consulta: https://serviciosportal.reniec.gob.pe/cetdnipi/inicio.htm
- Para consultar, deberá ingresar el número de su DNI.

Información mostrada por el portal:
- Fecha en que realizó el trámite.
- Local consular donde realizó el trámite.
- Avance del trámite.
- E

In [10]:
from app.rag.index_builder import crear_indice_faiss

total = crear_indice_faiss()

print(
    f"Índice FAISS reconstruido correctamente "
    f"con {total} fragmentos."
)

Índice FAISS reconstruido correctamente con 15 fragmentos.


In [9]:
from app.rag.retriever import recuperar_contexto

resultados = recuperar_contexto(
    "¿Qué requisitos necesito para renovar el DNI de mi menor hijo?"
)

print("Resultados:", len(resultados))

for r in resultados:
    print("----")
    print("Score:", r["score"])
    print("Archivo:", r["metadata"]["archivo"])
    print(r["contenido"][:700])

Resultados: 3
----
Score: 0.47144511342048645
Archivo: dni_menor_renovacion.txt
Condiciones especiales:
- Si el menor no tuvo antes DNI, el trámite que corresponde es "DNI de menor por primera vez".
- Si actúa un apoderado, deberá presentar el documento que sustente la custodia legal.

Cita:
- Renovación de DNI de menor de edad.
- URL: https://bookeo.com/cturin?type=223Y6PNNP174468CBDCD&option_TYPE_OF_SERVICE=YMRKCTKN

Advertencias:
- No acudir sin cita.
- Verificar la vigencia de los requisitos antes de la cita.
- Es imprescindible que las fotografías presentadas cumplan con las características indicadas.
----
Score: 0.6449533700942993
Archivo: dni_menor_renovacion.txt
Fuente: https://www.consulado.pe/es/turin/tramite/Paginas/DNI/Menor-renovacion.aspx
Fecha de consulta: 24/06/2026
Ámbito: Oficina consular / trámite Renovación de DNI de menor de edad

Trámite: Renovación de DNI de menor de edad.

Este documento corresponde únicamente al trámite de renovación de DNI de menor de edad. Si

In [10]:
from app.services.rag_service import RAGService

rag = RAGService()

resultado = rag.responder(
    "¿Qué requisitos necesito para renovar el DNI de mi menor hijo?"
)

print(resultado["respuesta"])
print("\nFuentes:")
for fuente in resultado["fuentes"]:
    print(fuente)

Para renovar el DNI de su menor hijo, necesita cumplir con los siguientes requisitos:

1. La presencia del menor y de un padre o apoderado legal, ambos con DNI original.
2. El DNI anterior del menor, si lo tuviera.
3. DNI vigente del padre, madre o representante legal.
4. Dos fotografías a color recientes, tamaño pasaporte, con fondo blanco, de frente, sin manchas, sin piercing, sin lentes, sin bufanda ni ningún elemento que cubra el rostro, no artísticas y sin sonrisa; diferentes a la que figura en el DNI actual.
5. Si necesita rectificar datos como nombres, apellidos, fecha o lugar de nacimiento, deberá presentar el acta de nacimiento del menor en original.
6. Pago de la tasa consular de €5.
7. Reserva de cita en el sistema oficial de la oficina consular.

Recuerde que es imprescindible asistir con cita previa y verificar la vigencia de estos requisitos antes de la cita.

Fuentes:
{'archivo': 'dni_menor_renovacion.txt', 'chunk': 2, 'score': 0.47144511342048645}
{'archivo': 'dni_menor

In [19]:
from app.rag.index_builder import crear_indice_faiss

total = crear_indice_faiss()
print("Fragmentos indexados:", total)

Fragmentos indexados: 11


In [20]:
from app.rag.retriever import recuperar_contexto

resultados = recuperar_contexto(
    "¿Qué requisitos necesito para sacar el DNI por primera vez de mi menor hijo?",
    k=5
)

for r in resultados:
    print("----")
    print("Archivo:", r["metadata"]["archivo"])
    print("Chunk:", r["metadata"]["chunk"])
    print("Distancia:", r["score"])
    print(r["contenido"][:1000])

----
Archivo: dni_menor_renovacion.txt
Chunk: 2
Distancia: 0.508354902267456
Condiciones especiales:
- Si el menor no tuvo antes DNI, el trámite que corresponde es "DNI de menor por primera vez".
- Si actúa un apoderado, deberá presentar el documento que sustente la custodia legal.

Cita:
- Renovación de DNI de menor de edad.
- URL: https://bookeo.com/cturin?type=223Y6PNNP174468CBDCD&option_TYPE_OF_SERVICE=YMRKCTKN

Advertencias:
- No acudir sin cita.
- Verificar la vigencia de los requisitos antes de la cita.
- Es imprescindible que las fotografías presentadas cumplan con las características indicadas.
----
Archivo: dni_menor_primera_vez.txt
Chunk: 3
Distancia: 0.6303791999816895
Costo:
- El costo del trámite de DNI de menor de edad por primera vez es de 5.00 euros.
- Este monto corresponde a la tasa consular indicada para el trámite.

Condiciones especiales:
- Si el menor ya tuvo DNI anteriormente, no corresponde este trámite; deberá solicitar la renovación de DNI de menor de edad.
-

In [2]:
from app.graph.dni_graph import construir_grafo_dni

grafo = construir_grafo_dni()

resultado = grafo.invoke({
    "pregunta": "¿Cuál es el costo del DNI de menor por primera vez?"
})

print("Intención:", resultado["intencion"])
print("Verificación:", resultado["verificacion"])
print("Respuesta:")
print(resultado["respuesta_final"])
print("\nFuentes:")
for fuente in resultado["fuentes"]:
    print(fuente)

Intención: dni_menor_primera_vez
Verificación: ok
Respuesta:
El costo del trámite de DNI de menor de edad por primera vez es de 5.00 euros. Este monto corresponde a la tasa consular indicada para el trámite.

Fuentes:
{'archivo': 'dni_menor_primera_vez.txt', 'chunk': 3, 'distancia': 0.34272557497024536}
{'archivo': 'dni_menor_primera_vez.txt', 'chunk': 0, 'distancia': 0.7152182459831238}
{'archivo': 'dni_menor_primera_vez.txt', 'chunk': 2, 'distancia': 0.8665359020233154}
{'archivo': 'dni_menor_primera_vez.txt', 'chunk': 4, 'distancia': 0.9664239883422852}


In [9]:
import inspect
import app.graph.dni_graph as dni_graph

print(inspect.getsource(dni_graph.clasificar_intencion))

def clasificar_intencion(state: DNIState) -> DNIState:
    pregunta = state["pregunta"]
    pregunta_lower = pregunta.lower()

    palabras_fuera_alcance = [
        "pasaporte",
        "visa",
        "visado",
        "carnet",
        "licencia",
        "apostilla",
        "poder",
        "partida",
        "acta",
        "matrimonio",
        "nacimiento",
        "defunción",
        "defuncion",
    ]

    palabras_estado = [
        "estado",
        "seguimiento",
        "avance",
        "ya está listo",
        "ya esta listo",
        "si está listo",
        "si esta listo",
        "consultar mi dni",
        "consultar el dni",
        "consultar trámite",
        "consultar tramite",
        "ver mi trámite",
        "ver mi tramite",
        "recoger mi dni",
        "entrega de dni",
        "ya llegó",
        "ya llego",
        "ficha",
        "reniec",
        "observado",
        "subsanado",
        "100%",
    ]

    palabras_primera_vez = [
        "prim

In [1]:
from app.graph.dni_graph import construir_grafo_dni

grafo = construir_grafo_dni()

resultado = grafo.invoke({
    "pregunta": "Necesito renovar mi pasaporte."
})

print("Intención:", resultado["intencion"])
print("Verificación:", resultado["verificacion"])
print("Respuesta:", resultado["respuesta_final"])
print("Fuentes:", resultado["fuentes"])

Intención: fuera_alcance
Verificación: ok
Respuesta: No cuento con información suficiente en las fuentes cargadas para responder sobre ese trámite. Este asistente piloto está limitado a consultas relacionadas con trámites de DNI cargados en la base documental.
Fuentes: []


In [2]:
from app.services.rag_service import RAGService

rag = RAGService()

resultado = rag.responder(
    "¿Cuál es el costo del DNI de menor por primera vez?"
)

print(resultado["intencion"])
print(resultado["verificacion"])
print(resultado["respuesta"])

dni_menor_primera_vez
ok
El costo del trámite de DNI de menor de edad por primera vez es de 5.00 euros. Este monto corresponde a la tasa consular indicada para el trámite.


In [3]:
from pathlib import Path

ruta_log = Path("data/logs/consultas_log.csv")
print(ruta_log.exists())
print(ruta_log)

True
data\logs\consultas_log.csv


In [5]:
from app.services.rag_service import RAGService

rag = RAGService()

preguntas = [
    "¿Qué requisitos necesito para renovar el DNI de mi menor hijo?",
    "Necesito sacar una cita para el renuevo de DNI de mi menor hijo.",
    "¿Qué requisitos necesito para tramitar el DNI de mi menor hijo por primera vez?",
    "¿Cuál es el costo del DNI de menor por primera vez?",
    "¿Puedo darte mi número de DNI para que consultes si ya llegó?",
    "Necesito renovar mi pasaporte.",
]

for pregunta in preguntas:
    resultado = rag.responder(pregunta)
    print("=" * 80)
    print("Pregunta:", pregunta)
    print("Intención:", resultado["intencion"])
    print("Verificación:", resultado["verificacion"])
    print("Fuentes:", resultado["fuentes"])

Pregunta: ¿Qué requisitos necesito para renovar el DNI de mi menor hijo?
Intención: dni_menor_renovacion
Verificación: ok
Fuentes: [{'archivo': 'dni_menor_renovacion.txt', 'chunk': 2, 'distancia': 0.47144511342048645}, {'archivo': 'dni_menor_renovacion.txt', 'chunk': 0, 'distancia': 0.6452489495277405}, {'archivo': 'dni_menor_renovacion.txt', 'chunk': 1, 'distancia': 0.7841145992279053}]
Pregunta: Necesito sacar una cita para el renuevo de DNI de mi menor hijo.
Intención: dni_menor_renovacion
Verificación: ok
Fuentes: [{'archivo': 'dni_menor_renovacion.txt', 'chunk': 2, 'distancia': 0.5774894952774048}, {'archivo': 'dni_menor_renovacion.txt', 'chunk': 1, 'distancia': 0.8789634108543396}, {'archivo': 'dni_menor_renovacion.txt', 'chunk': 0, 'distancia': 0.9042848348617554}]
Pregunta: ¿Qué requisitos necesito para tramitar el DNI de mi menor hijo por primera vez?
Intención: dni_menor_primera_vez
Verificación: ok
Fuentes: [{'archivo': 'dni_menor_primera_vez.txt', 'chunk': 3, 'distancia': 0

In [6]:
import pandas as pd

df = pd.read_csv("data/logs/consultas_log.csv")
df.tail()

,fecha_hora,pregunta,intencion,verificacion,respuesta,fuentes
2,2026-07-04T17:12:09,Necesito sacar una cita para el renuevo de DNI...,dni_menor_renovacion,ok,Para sacar una cita para la renovación del DNI...,"[{""archivo"": ""dni_menor_renovacion.txt"", ""chun..."
3,2026-07-04T17:12:14,¿Qué requisitos necesito para tramitar el DNI ...,dni_menor_primera_vez,ok,Para tramitar el DNI de su menor hijo por prim...,"[{""archivo"": ""dni_menor_primera_vez.txt"", ""chu..."
4,2026-07-04T17:12:16,¿Cuál es el costo del DNI de menor por primera...,dni_menor_primera_vez,ok,El costo del trámite de DNI de menor de edad p...,"[{""archivo"": ""dni_menor_primera_vez.txt"", ""chu..."
5,2026-07-04T17:12:18,¿Puedo darte mi número de DNI para que consult...,dni_consulta_estado,ok,El asistente no puede recibir ni consultar núm...,"[{""archivo"": ""dni_consulta_estado.txt"", ""chunk..."
6,2026-07-04T17:12:18,Necesito renovar mi pasaporte.,fuera_alcance,ok,No cuento con información suficiente en las fu...,[]


In [7]:
from app.services.evaluation_service import generar_resumen_evaluacion

resumen = generar_resumen_evaluacion()

print("Total de consultas:", resumen["total_consultas"])
print("Consultas fuera de alcance:", resumen["consultas_fuera_alcance"])
print("Consultas observadas:", resumen["consultas_observadas"])

resumen["resumen_intenciones"]

Total de consultas: 7
Consultas fuera de alcance: 1
Consultas observadas: 0


,cantidad,count
0,dni_menor_primera_vez,3
1,dni_menor_renovacion,2
2,dni_consulta_estado,1
3,fuera_alcance,1


In [8]:
resumen["resumen_verificacion"]

,cantidad,count
0,ok,7


In [9]:
resumen["detalle"].tail(10)

,fecha_hora,pregunta,intencion,verificacion,respuesta,fuentes
0,2026-07-04T17:10:04,¿Cuál es el costo del DNI de menor por primera...,dni_menor_primera_vez,ok,El costo del trámite de DNI de menor de edad p...,"[{""archivo"": ""dni_menor_primera_vez.txt"", ""chu..."
1,2026-07-04T17:12:07,¿Qué requisitos necesito para renovar el DNI d...,dni_menor_renovacion,ok,"Para la renovación del DNI de su menor hijo, d...","[{""archivo"": ""dni_menor_renovacion.txt"", ""chun..."
2,2026-07-04T17:12:09,Necesito sacar una cita para el renuevo de DNI...,dni_menor_renovacion,ok,Para sacar una cita para la renovación del DNI...,"[{""archivo"": ""dni_menor_renovacion.txt"", ""chun..."
3,2026-07-04T17:12:14,¿Qué requisitos necesito para tramitar el DNI ...,dni_menor_primera_vez,ok,Para tramitar el DNI de su menor hijo por prim...,"[{""archivo"": ""dni_menor_primera_vez.txt"", ""chu..."
4,2026-07-04T17:12:16,¿Cuál es el costo del DNI de menor por primera...,dni_menor_primera_vez,ok,El costo del trámite de DNI de menor de edad p...,"[{""archivo"": ""dni_menor_primera_vez.txt"", ""chu..."
5,2026-07-04T17:12:18,¿Puedo darte mi número de DNI para que consult...,dni_consulta_estado,ok,El asistente no puede recibir ni consultar núm...,"[{""archivo"": ""dni_consulta_estado.txt"", ""chunk..."
6,2026-07-04T17:12:18,Necesito renovar mi pasaporte.,fuera_alcance,ok,No cuento con información suficiente en las fu...,[]


In [10]:
from app.services.evaluation_service import generar_resumen_fuentes

df_fuentes = generar_resumen_fuentes()
df_fuentes

,cantidad,count
0,dni_menor_primera_vez.txt,12
1,dni_menor_renovacion.txt,6
2,dni_consulta_estado.txt,3


In [11]:
import pandas as pd

casos_prueba = pd.DataFrame(
    [
        {
            "codigo": "P01",
            "caso": "Requisitos renovación DNI menor",
            "pregunta": "¿Qué requisitos necesito para renovar el DNI de mi menor hijo?",
            "intencion_esperada": "dni_menor_renovacion",
        },
        {
            "codigo": "P02",
            "caso": "Cita renovación DNI menor",
            "pregunta": "Necesito sacar una cita para el renuevo de DNI de mi menor hijo.",
            "intencion_esperada": "dni_menor_renovacion",
        },
        {
            "codigo": "P03",
            "caso": "DNI menor por primera vez",
            "pregunta": "¿Qué requisitos necesito para tramitar el DNI de mi menor hijo por primera vez?",
            "intencion_esperada": "dni_menor_primera_vez",
        },
        {
            "codigo": "P04",
            "caso": "Costo DNI menor primera vez",
            "pregunta": "¿Cuál es el costo del DNI de menor por primera vez?",
            "intencion_esperada": "dni_menor_primera_vez",
        },
        {
            "codigo": "P05",
            "caso": "Consulta de estado",
            "pregunta": "¿Puedo darte mi número de DNI para que consultes si ya llegó?",
            "intencion_esperada": "dni_consulta_estado",
        },
        {
            "codigo": "P06",
            "caso": "Fuera de alcance",
            "pregunta": "Necesito renovar mi pasaporte.",
            "intencion_esperada": "fuera_alcance",
        },
    ]
)

casos_prueba

,codigo,caso,pregunta,intencion_esperada
0,P01,Requisitos renovación DNI menor,¿Qué requisitos necesito para renovar el DNI d...,dni_menor_renovacion
1,P02,Cita renovación DNI menor,Necesito sacar una cita para el renuevo de DNI...,dni_menor_renovacion
2,P03,DNI menor por primera vez,¿Qué requisitos necesito para tramitar el DNI ...,dni_menor_primera_vez
3,P04,Costo DNI menor primera vez,¿Cuál es el costo del DNI de menor por primera...,dni_menor_primera_vez
4,P05,Consulta de estado,¿Puedo darte mi número de DNI para que consult...,dni_consulta_estado
5,P06,Fuera de alcance,Necesito renovar mi pasaporte.,fuera_alcance


In [12]:
from app.services.rag_service import RAGService

rag = RAGService()

resultados = []

for _, caso in casos_prueba.iterrows():
    resultado = rag.responder(caso["pregunta"])

    resultados.append(
        {
            "codigo": caso["codigo"],
            "caso": caso["caso"],
            "pregunta": caso["pregunta"],
            "intencion_esperada": caso["intencion_esperada"],
            "intencion_obtenida": resultado["intencion"],
            "verificacion": resultado["verificacion"],
            "cumple_intencion": caso["intencion_esperada"] == resultado["intencion"],
            "numero_fuentes": len(resultado["fuentes"]),
        }
    )

df_resultados = pd.DataFrame(resultados)
df_resultados

,codigo,caso,pregunta,intencion_esperada,intencion_obtenida,verificacion,cumple_intencion,numero_fuentes
0,P01,Requisitos renovación DNI menor,¿Qué requisitos necesito para renovar el DNI d...,dni_menor_renovacion,dni_menor_renovacion,ok,True,3
1,P02,Cita renovación DNI menor,Necesito sacar una cita para el renuevo de DNI...,dni_menor_renovacion,dni_menor_renovacion,ok,True,3
2,P03,DNI menor por primera vez,¿Qué requisitos necesito para tramitar el DNI ...,dni_menor_primera_vez,dni_menor_primera_vez,ok,True,4
3,P04,Costo DNI menor primera vez,¿Cuál es el costo del DNI de menor por primera...,dni_menor_primera_vez,dni_menor_primera_vez,ok,True,4
4,P05,Consulta de estado,¿Puedo darte mi número de DNI para que consult...,dni_consulta_estado,dni_consulta_estado,ok,True,3
5,P06,Fuera de alcance,Necesito renovar mi pasaporte.,fuera_alcance,fuera_alcance,ok,True,0


In [13]:
ruta_resultados = "data/logs/evaluacion_casos_prueba.csv"

df_resultados.to_csv(
    ruta_resultados,
    index=False,
    encoding="utf-8"
)

print("Archivo generado:", ruta_resultados)

Archivo generado: data/logs/evaluacion_casos_prueba.csv
